# 05 — Feature engineering

This notebook turns canonical PON telemetry into causal, metric-aware model
features. It uses the catalogue frozen before modelling: gauges preserve their
units, zero-inflated BER and CRC values use hurdle features, counts use
non-negative transforms, and cumulative counters are differenced without
bridging resets or collection gaps.

Only calibration and development telemetry are opened. No fault, ticket or
holdout label is read.


## 1. Setup and frozen inputs


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import shutil
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.detectors import (
    iter_episode_frames,
    materialize_measurement_features,
    materialize_wide_partition,
    measurement_features,
)
from telco_anomaly.features import feature_policy
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    load_config,
    read_json,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Feature engineering is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_eda = os.getenv("PON_EDA_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_features = os.getenv("PON_FEATURE_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv("TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET])
EDA_RUN_ID = os.getenv(
    "TELCO_EDA_RUN_ID", legacy_eda or f"{DATASET}_calibration_eda_v2"
)
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", legacy_features or f"{DATASET}_features_v2"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
SPLIT_ROOT = RUN_ROOT / "SPLITS"
EDA_ROOT = DATA_ROOT / "eda" / DATASET / EDA_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID

for required in (CORE_ROOT, SPLIT_ROOT, EDA_ROOT):
    if not required.exists():
        raise FileNotFoundError(f"Missing prerequisite {required}")
if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Feature engineering must use the truth-unmounted run")

catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
core_manifest = read_json(CORE_ROOT / "manifest.json")
eda_manifest = read_json(EDA_ROOT / "eda_manifest.json")
eda_decisions = read_json(EDA_ROOT / "eda_decisions.json")
assert eda_manifest["partition"] == "calibration"
assert eda_manifest["core_fingerprint"] == core_manifest["fingerprint"]
assert eda_decisions["canonical_fingerprint"] == core_manifest["fingerprint"]

HISTORY_SECONDS = int(os.getenv(
    "PON_HISTORY_SECONDS", eda_decisions["history_window_seconds"]
))
MINIMUM_HISTORY_SECONDS = int(os.getenv(
    "PON_MIN_HISTORY_SECONDS", eda_decisions["minimum_history_seconds"]
))
DISPERSION_WINDOW_SECONDS = int(os.getenv(
    "PON_DISPERSION_SECONDS", eda_decisions["dispersion_window_seconds"]
))
SEASONAL_PERIODS = {
    metric_id: period
    for metric_id, period in eda_decisions["seasonality_decisions"].items()
    if period is not None
}
GAP_TOLERANCE = 1.5

display(pd.Series({
    "canonical_input": str(CORE_ROOT),
    "dataset": DATASET,
    "calibration_eda": str(EDA_ROOT),
    "feature_output": str(OUTPUT_ROOT),
    "history_days": HISTORY_SECONDS / 86_400,
    "minimum_history_days": MINIMUM_HISTORY_SECONDS / 86_400,
    "dispersion_hours": DISPERSION_WINDOW_SECONDS / 3600,
    "seasonal_metrics": len(SEASONAL_PERIODS),
}, name="value").to_frame())


## 2. Inspect the feature policy

The table below is the auditable bridge from semantic metrics to features.
`asset_health` features may enter a detector. Clipping flags remain
`data_quality`; they are retained for monitoring but cannot become health
evidence.


In [ ]:
policy = feature_policy(catalogue)
display(policy)

assert set(policy.loc[policy["feature"].str.endswith("__clipped"), "role"]) == {
    "data_quality"
}
assert not policy["feature"].str.startswith(("gt_", "truth_", "fault_"), na=False).any()


## 3. Materialise calibration and development features


In [ ]:
def build_partition(partition, output):
    wide_path = output / f"{partition}_wide.parquet"
    feature_path = output / f"{partition}_features.parquet"
    definition = materialize_wide_partition(
        CORE_ROOT,
        SPLIT_ROOT,
        partition,
        catalogue,
        wide_path,
        lookback_seconds=HISTORY_SECONDS,
        memory_limit=os.getenv("DUCKDB_MEMORY_LIMIT", "2GB"),
        threads=int(os.getenv("DUCKDB_THREADS", "2")),
    )
    materialize_measurement_features(
        wide_path,
        catalogue,
        feature_path,
        history_window_seconds=HISTORY_SECONDS,
        minimum_history_seconds=MINIMUM_HISTORY_SECONDS,
        gap_tolerance=GAP_TOLERANCE,
        seasonal_periods=SEASONAL_PERIODS,
        score_start=definition["score_start"],
        score_end=definition["score_end"],
    )
    return {
        "wide": wide_path.name,
        "features": feature_path.name,
        "rows": pq.ParquetFile(feature_path).metadata.num_rows,
    }


if OUTPUT_ROOT.exists():
    feature_manifest = read_json(OUTPUT_ROOT / "feature_manifest.json")
    if feature_manifest["core_fingerprint"] != core_manifest["fingerprint"]:
        raise ValueError("Existing features belong to a different canonical run")
    print("Using existing immutable features:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        partitions = {
            name: build_partition(name, output)
            for name in ("calibration", "development")
        }
        policy.to_parquet(output / "feature_policy.parquet", index=False)
        feature_manifest = {
            "dataset": DATASET,
            "core_fingerprint": core_manifest["fingerprint"],
            "eda_manifest_sha256": file_sha256(EDA_ROOT / "eda_manifest.json"),
            "history_seconds": HISTORY_SECONDS,
            "minimum_history_seconds": MINIMUM_HISTORY_SECONDS,
            "dispersion_window_seconds": DISPERSION_WINDOW_SECONDS,
            "seasonal_periods": SEASONAL_PERIODS,
            "gap_tolerance": GAP_TOLERANCE,
            "partitions": partitions,
            "truth_files_read": [],
            "holdout_materialised": False,
        }
        write_json(output / "feature_manifest.json", feature_manifest)
    print("Saved:", OUTPUT_ROOT)

display(pd.Series(feature_manifest["partitions"], name="partition evidence").to_frame())


## 4. Causality test

For one complete calibration episode, features calculated on a time prefix
must equal the corresponding prefix calculated when later rows are present.
This catches centred windows, backward filling and other future leakage.


In [ ]:
calibration_wide = OUTPUT_ROOT / feature_manifest["partitions"]["calibration"]["wide"]
episode = next(iter_episode_frames(calibration_wide))
if len(episode) < 20:
    raise ValueError("A calibration episode is too short for the causality test")

cutoff = max(10, len(episode) // 2)
full = measurement_features(
    episode,
    catalogue,
    history_window_seconds=HISTORY_SECONDS,
    minimum_history_seconds=MINIMUM_HISTORY_SECONDS,
    gap_tolerance=GAP_TOLERANCE,
    seasonal_periods=SEASONAL_PERIODS,
).iloc[:cutoff].reset_index(drop=True)
prefix = measurement_features(
    episode.iloc[:cutoff].copy(),
    catalogue,
    history_window_seconds=HISTORY_SECONDS,
    minimum_history_seconds=MINIMUM_HISTORY_SECONDS,
    gap_tolerance=GAP_TOLERANCE,
    seasonal_periods=SEASONAL_PERIODS,
).reset_index(drop=True)

pd.testing.assert_frame_equal(full, prefix, check_exact=True)
print("PASS — later observations cannot change earlier feature values")


## 5. Feature sample and acceptance


In [ ]:
feature_path = OUTPUT_ROOT / feature_manifest["partitions"]["calibration"]["features"]
sample = next(pq.ParquetFile(feature_path).iter_batches(batch_size=10)).to_pandas()
display(sample)

health_columns = policy.loc[policy["role"].eq("asset_health"), "feature"]
available = sorted(
    set(health_columns) & set(pq.ParquetFile(feature_path).schema_arrow.names)
)
display(pd.DataFrame({"model_feature": available}))

assert feature_manifest["truth_files_read"] == []
assert feature_manifest["holdout_materialised"] is False
assert available
print("PASS — causal model features are ready")
print("Next: 06_PRIMARY_UNSUPERVISED_MODEL.ipynb")
